# harness-admission: clean Colab run
Run the cells **in order, one at a time**. Every step is resumable: after a disconnect or restart, run **cell 1** again, then the step you were on. Finished work is skipped.

Rules:
- **Never press stop** while a model is loading or generating.
- **Never load a model inside the notebook** (`LLM(...)`). All model runs go through `run(...)`, which frees the GPU first.
- Keep the same GPU type for the whole experiment (cell 1 checks this).

In [ ]:
# ===== CELL 1: setup (run this first, and again after every restart) =====
import os, re, subprocess, sys, time, json, glob

REPO_URL = "https://github.com/Soumya080/harness-admission-colab.git"
REPO = os.environ.get("HA_REPO_DIR", "/content/ha")

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{cmd}\n{r.stdout[-2000:]}\n{r.stderr[-2000:]}")
    return r.stdout.strip()

# 1. packages (only installs if vLLM is missing)
try:
    import vllm  # noqa
except ImportError:
    print("installing packages (5-10 min)...")
    sh(f"{sys.executable} -m pip install -q vllm datasets pandas pyarrow scipy matplotlib pytest")
    print("installed. If the next cells fail on import, do Runtime > Restart session and run this cell again.")

# 2. code
if os.path.isdir(os.path.join(REPO, ".git")):
    sh(f"cd {REPO} && git pull -q", check=False)
elif not os.path.isdir(REPO):
    sh(f"git clone -q {REPO_URL} {REPO}")
os.chdir(REPO)
assert os.path.isdir("tests") and os.path.isdir("scripts"), "wrong repo: tests/ or scripts/ missing"

# 3. where results live: Google Drive if it mounts, else local disk
if "HA_HOME" not in os.environ or not os.path.isdir(os.environ["HA_HOME"]):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive not mounted:", str(e)[:80])
    home = "/content/drive/MyDrive/ha_home" if os.path.isdir("/content/drive/MyDrive") else "/content/ha_home"
    os.makedirs(home, exist_ok=True)
    os.environ["HA_HOME"] = home
HOME = os.environ["HA_HOME"]
ON_DRIVE = os.path.ismount("/content/drive") and HOME.startswith("/content/drive")
os.environ.setdefault("HA_DTYPE", "half")
os.environ.setdefault("HA_GPU_UTIL", "0.85")
os.makedirs(f"{HOME}/logs", exist_ok=True); os.makedirs(f"{HOME}/results", exist_ok=True)

# 4. GPU helpers
def gpu_name():
    try: return sh("nvidia-smi --query-gpu=name --format=csv,noheader").splitlines()[0].strip()
    except Exception: return "NO GPU"

def gpu_used():
    try: return int(sh("nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits").split()[0])
    except Exception: return 0

def free_gpu(limit=1500):
    """Kill every leftover process holding GPU memory. Raises if the GPU cannot be freed."""
    if gpu_used() < limit: return
    pids = sh("nvidia-smi --query-compute-apps=pid --format=csv,noheader", check=False).split()
    for p in pids:
        if p.isdigit() and int(p) != os.getpid():
            sh(f"kill -9 {p}", check=False)
    for pat in ("VLLM", "EngineCore", "scripts.run_grid", "proposer.propose"):
        sh(f"pkill -9 -f '{pat}'", check=False)
    for _ in range(10):
        time.sleep(2)
        if gpu_used() < limit: print("GPU freed."); return
    raise RuntimeError(f"GPU still has {gpu_used()} MiB in use. Do Runtime > Restart session, "
                       "then run cell 1 again. (A model was loaded inside the notebook.)")

NOISE = re.compile(r"EngineCore pid|MainProcess pid|torch/_|Downloading|Reconstruct|Fetching|safetensors|"
                   r"unauthenticated|it/s|Generating .* split|README.md|\.parquet:|config\.json|Download complete")

def run(cmd, gpu=True, save=None):
    """Run a repo command in its own process. Shows only the useful lines; full log goes to HA_HOME/logs."""
    if gpu: free_gpu()
    log = f"{HOME}/logs/{time.strftime('%m%d_%H%M%S')}_{re.sub(r'[^a-z_]+', '_', cmd.lower())[:40]}.log"
    print(f"$ {cmd}")
    t0, shown = time.time(), []
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         cwd=REPO, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    with open(log, "w") as f:
        for line in p.stdout:
            f.write(line)
            if line.strip() and not NOISE.search(line):
                print("  " + line.rstrip()[:200]); shown.append(line)
    p.wait()
    if save:
        open(f"{HOME}/results/{save}", "w").writelines(shown)
    if p.returncode != 0:
        errs = [l.strip() for l in open(log) if re.search(r"Error|STOP|FAILED", l)]
        print("\nFAILED. Root cause lines:"); [print("  ", e[-300:]) for e in errs[-4:]]
        raise RuntimeError(f"step failed (full log: {log})")
    print(f"ok ({time.time() - t0:.0f}s)\n")

def harness_tags():
    out = {}
    for f in glob.glob(f"{HOME}/cache/harnesses/*.json"):
        t = json.load(open(f)).get("tag", "?"); out[t] = out.get(t, 0) + 1
    return out

# 5. one GPU type for the whole experiment
name = gpu_name(); gfile = f"{HOME}/gpu_name.txt"
if os.path.exists(gfile):
    old = open(gfile).read().strip()
    if old != name:
        print(f"!!! WARNING: earlier results were made on '{old}', this session has '{name}'.\n"
              f"!!! Do not continue the grid on a different GPU. Get '{old}' again, or delete {HOME}/cache/outcomes and rerun.")
elif name != "NO GPU":
    open(gfile, "w").write(name)

n_out = len(glob.glob(f"{HOME}/cache/outcomes/*.parquet"))
print(f"repo: {REPO}\nresults folder: {HOME}  ({'Google Drive: survives disconnects' if ON_DRIVE else 'LOCAL DISK: run the backup cell often'})")
print(f"GPU: {name}, {gpu_used()} MiB in use\nharnesses so far: {harness_tags()} | finished result files: {n_out}")

In [ ]:
# ===== CELL 2: CPU checks (about 5 min). Must end with "ALL DONE". =====
run("python -m pytest -q tests", gpu=False)
run("HA_FAKE=1 HA_HOME=$(mktemp -d) bash scripts/run_all.sh 2>&1 | tail -3", gpu=False)

In [ ]:
# ===== CELL 3: base + planted + STOP CHECK (GPU, about 10-20 min) =====
if not os.path.exists(f"{HOME}/configs/pools_gsm8k.json"):
    run("python -m data.pools", gpu=False)
run("python -m proposer.manual", gpu=False)
run("python -m scripts.run_grid --only base")
run("python -m scripts.run_grid --only planted")
run("python -m scripts.check_stop", gpu=False)      # raises if the check fails

**Continue only if cell 3 printed `CHECK PASSED`.**

In [ ]:
# ===== CELL 4: proposer, 30 LLM edits (GPU, 7B model, about 10-20 min) =====
n = harness_tags().get("llm", 0)
print("llm edits so far:", n)
if n < 30:
    run(f"python -m proposer.propose --n {30 - n}")
print(harness_tags())      # expect base 1, planted 1, noop 5, llm 30

In [ ]:
# ===== CELL 5: full grid (GPU, the long step). Rerun this cell if it stops. =====
run("python -m scripts.run_grid")
run("python -m scripts.diagnose", gpu=False, save="diagnose.txt")

In [ ]:
# ===== CELL 6: stage-0 simulations + report + ablations (CPU, about 20-40 min) =====
run("python -m sims.stage0", gpu=False, save="stage0.txt")
run("python -m sims.plot_stage0", gpu=False)
run("python -m scripts.make_report --runs 1000 --ablations", gpu=False, save="report.txt")
print(sorted(os.listdir(f"{HOME}/results")))

In [ ]:
# ===== CELL 7: backup. Run after cells 3, 4, 5 and 6 if results are on LOCAL DISK. =====
zip_path = "/content/ha_home_backup.zip"
sh(f"cd {os.path.dirname(HOME)} && zip -qr {zip_path} {os.path.basename(HOME)}")
print("backup:", zip_path, f"{os.path.getsize(zip_path)/1e6:.1f} MB")
try:
    from google.colab import files; files.download(zip_path)
except Exception as e:
    print("download it from the file sidebar:", e)

### Restoring a backup in a new session
Upload `ha_home_backup.zip` with the file sidebar, then run:
```python
import subprocess; subprocess.run("mkdir -p /content && cd /content && unzip -qo ha_home_backup.zip", shell=True)
```
Then run cell 1 and continue from the step you were on.